In [ ]:
import json,pickle,zipfile,re,math,os,time
import numpy as np
import unicodedata
import torch, torch.nn as nn, torch.nn.functional as F
from peft import LoraConfig,AdaLoraConfig, get_peft_model
from collections import Counter
from tqdm import tqdm
from tqdm.auto import tqdm
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient
from torch.utils.data import DataLoader
from torch.utils import checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

In [ ]:
## tokenizer
 
WORD_END = 256
 
def pre_tokenize_preserve(text):
    return re.findall(r"\w+|[^\w\s]|\s+", text)
 
def word_to_symbols(word):
    byte_values = word.encode("utf-8")
    symbols = [(b,) for b in byte_values]
    symbols.append((WORD_END,))
    return tuple(symbols)
 
def get_adjacent_pairs(symbols):
    return set(zip(symbols, symbols[1:]))
 
def apply_bpe_merges(symbols, merge_rank):
    symbols = list(symbols)
    while len(symbols) > 1:
        pairs = get_adjacent_pairs(symbols)
        candidates = [p for p in pairs if p in merge_rank]
        if not candidates:
            break
        best_pair = min(candidates, key=lambda p: merge_rank[p])
        sa, sb = best_pair
        merged = sa + sb
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == sa and symbols[i + 1] == sb:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols
 
def load_tokenizer(pkl_path):
    bpe_data = pickle.load(open(pkl_path, "rb"))
    stoi, merge_rank = bpe_data["stoi"], bpe_data["merge_rank"]
    cache = {}
 
    def encode_unit(unit):
        if unit not in cache:
            cache[unit] = [stoi[s] for s in apply_bpe_merges(word_to_symbols(unit), merge_rank)]
        return cache[unit]
 
    def encode(text):
        ids = []
        for unit in pre_tokenize_preserve(text):
            ids += encode_unit(" " if unit.isspace() else unit)
        return ids
 
    return bpe_data, encode
 
 
def clean_text(t):
    t = unicodedata.normalize("NFKC", t)
    t = re.sub(r"(\w)-[ \t]*\n[ \t]*(\w)", r"\1\2", t)
    return re.sub(r"\s+", " ", t).strip()

# MODEL V2

In [ ]:
class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE requires an even head_dim"
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_seq_len = max_seq_len
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, context_length, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.context_length = context_length
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if attn_mask is not None:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=dropout_p)
        elif past_kv is not None:
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p)
        else:
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, context_length, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, context_length, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        attn_out, present = self.attn(self.ln_1(x), attn_mask, past_kv, use_cache, position_offset)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return (x, present) if use_cache else x

class DecoderOnlyLM_RoPE(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head,
                dropout=0.1, pad_id=0, gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.gradient_checkpointing = gradient_checkpointing
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, context_length, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias)
            nn.init.ones_(module.weight)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        combined_mask = None
        if attention_mask is not None:
            key_mask = attention_mask.bool()[:, None, None, :]
            if past_len == 0:
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
                eye = torch.eye(T, dtype=torch.bool, device=input_ids.device)
                combined_mask = (causal[None, None] & key_mask) | eye[None, None]
            else:
                combined_mask = key_mask

        x = self.drop(self.wte(input_ids))

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None
            if self.gradient_checkpointing and self.training and not use_cache:
                x = checkpoint.checkpoint(block, x, combined_mask, None, False, past_len,
                                          use_reentrant=False)
            else:
                out = block(x, combined_mask, past, use_cache, past_len)
                if use_cache:
                    x, present = out
                    presents.append(present)
                else:
                    x = out

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=attention_mask,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

# MODEL V5

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)

class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

        k_exp = k.repeat_interleave(self.n_rep, dim=1)
        v_exp = v.repeat_interleave(self.n_rep, dim=1)

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if attn_mask is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)
        elif past_kv is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p)
        else:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present

class Expert(nn.Module):
    def __init__(self, n_embd, expert_hidden_mult=4, dropout=0.1):
        super().__init__()
        hidden = expert_hidden_mult * n_embd
        self.c_fc = nn.Linear(n_embd, hidden)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(hidden, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class MoEFeedForward(nn.Module):
    def __init__(self, n_embd, num_experts=4, expert_hidden_mult=4, dropout=0.1):
        super().__init__()
        self.num_experts = num_experts
        self.router = nn.Linear(n_embd, num_experts, bias=False)
        self.experts = nn.ModuleList([
            Expert(n_embd, expert_hidden_mult, dropout) for _ in range(num_experts)
        ])

    def forward(self, x):
        B, T, C = x.shape
        x_flat = x.view(-1, C)                                  

        router_logits = self.router(x_flat)                     
        router_probs = F.softmax(router_logits, dim=-1)
        top1_prob, top1_idx = router_probs.max(dim=-1)          
        one_hot = F.one_hot(top1_idx, num_classes=self.num_experts).to(x.dtype)   

        expert_outputs = torch.stack([expert(x_flat) for expert in self.experts], dim=1)  
        gate = (one_hot * top1_prob.unsqueeze(-1)).unsqueeze(-1)                          
        output = (expert_outputs * gate).sum(dim=1)                                        
        with torch.no_grad():
            tokens_per_expert = one_hot.float().mean(dim=0)      
        mean_router_prob = router_probs.mean(dim=0)              
        aux_loss = self.num_experts * (tokens_per_expert * mean_router_prob).sum()

        return output.view(B, T, C), aux_loss

class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, num_experts, dropout=0.1):
        super().__init__()
        self.ln_1_pre = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, context_length, dropout)
        self.ln_1_post = RMSNorm(n_embd)

        self.ln_2_pre = RMSNorm(n_embd)
        self.moe = MoEFeedForward(n_embd, num_experts=num_experts, dropout=dropout)
        self.ln_2_post = RMSNorm(n_embd)

    def forward_train(self, x, attn_mask):
        attn_out, _ = self.attn(self.ln_1_pre(x), attn_mask)
        x = x + self.ln_1_post(attn_out)

        moe_out, aux_loss = self.moe(self.ln_2_pre(x))
        x = x + self.ln_2_post(moe_out)
        return x, aux_loss

    def forward_infer(self, x, past_kv, use_cache, position_offset):
        attn_out, present = self.attn(self.ln_1_pre(x), None, past_kv, use_cache, position_offset)
        x = x + self.ln_1_post(attn_out)

        moe_out, _ = self.moe(self.ln_2_pre(x))
        x = x + self.ln_2_post(moe_out)
        return x, present

class DecoderOnlyLM_MoE_Sandwich(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head, n_kv_head,num_experts=4, aux_loss_weight=0.01, dropout=0.1, pad_id=0,gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.aux_loss_weight = aux_loss_weight
        self.gradient_checkpointing = gradient_checkpointing

        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, n_kv_head, context_length, num_experts, dropout)
            for _ in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        x = self.drop(self.wte(input_ids))

        if use_cache:
            presents = []
            for i, block in enumerate(self.blocks):
                past = past_key_values[i] if past_key_values is not None else None
                x, present = block.forward_infer(x, past, use_cache, past_len)
                presents.append(present)
        else:
            combined_mask = None
            if attention_mask is not None:
                key_mask = attention_mask.bool()[:, None, None, :]
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
                eye = torch.eye(T, dtype=torch.bool, device=input_ids.device)
                combined_mask = (causal[None, None] & key_mask) | eye[None, None]

            aux_losses = []
            for block in self.blocks:
                if self.gradient_checkpointing and self.training:
                    x, aux = checkpoint.checkpoint(block.forward_train, x, combined_mask,
                                                   use_reentrant=False)
                else:
                    x, aux = block.forward_train(x, combined_mask)
                aux_losses.append(aux)
            presents = None

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            ce_loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
            aux_loss = torch.stack(aux_losses).mean() if not use_cache else torch.tensor(0.0, device=x.device)
            loss = ce_loss + self.aux_loss_weight * aux_loss

        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=None,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def count_active_parameters(model, num_experts):
    total = count_parameters(model)
    expert_params = sum(p.numel() for p in model.blocks[0].moe.experts[0].parameters())
    num_layers = len(model.blocks)
    unused_experts_params = expert_params * (num_experts - 1) * num_layers
    return total - unused_experts_params

# MODEL V6

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)


def is_global_layer(layer_idx, n_layer, global_every_n):
    return (n_layer - 1 - layer_idx) % global_every_n == 0


class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, window_size=None, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = n_embd // n_head
        self.window_size = window_size   

        self.q_proj = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def _build_mask(self, T, device, key_padding_mask=None):
        i = torch.arange(T, device=device)
        if self.window_size is None:
            causal = i[:, None] >= i[None, :]                              
        else:
            diff = i[:, None] - i[None, :]
            causal = (diff >= 0) & (diff < self.window_size)               
        mask = causal[None, None, :, :]
        if key_padding_mask is not None:
            key_mask = key_padding_mask.bool()[:, None, None, :]
            eye = torch.eye(T, dtype=torch.bool, device=device)
            mask = (mask & key_mask) | eye[None, None, :, :]
        return mask

    def forward(self, x, key_padding_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
            if self.window_size is not None and k.size(2) > self.window_size:
                k = k[:, :, -self.window_size:, :]
                v = v[:, :, -self.window_size:, :]
                if key_padding_mask is not None:
                    key_padding_mask = key_padding_mask[:, -self.window_size:]
        present = (k, v) if use_cache else None

        k_exp = k.repeat_interleave(self.n_rep, dim=1)
        v_exp = v.repeat_interleave(self.n_rep, dim=1)

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if past_kv is None and T > 1:
            attn_mask = self._build_mask(T, x.device, key_padding_mask)
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)
        else:
            attn_mask = None
            if key_padding_mask is not None:
                attn_mask = key_padding_mask.bool()[:, None, None, :]
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, window_size, dropout=0.1):
        super().__init__()
        self.ln_1_pre = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, context_length, window_size, dropout)
        self.ln_1_post = RMSNorm(n_embd)
        self.ln_2_pre = RMSNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)
        self.ln_2_post = RMSNorm(n_embd)

    def forward(self, x, key_padding_mask=None, past_kv=None, use_cache=False, position_offset=0):
        attn_out, present = self.attn(self.ln_1_pre(x), key_padding_mask, past_kv, use_cache, position_offset)
        x = x + self.ln_1_post(attn_out)
        x = x + self.ln_2_post(self.mlp(self.ln_2_pre(x)))
        return (x, present) if use_cache else x

class DecoderOnlyLM_HybridMask(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head, n_kv_head,
                window_size=192, global_every_n=4, dropout=0.1, pad_id=0,
                gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.gradient_checkpointing = gradient_checkpointing
        self.window_size = window_size
        self.global_every_n = global_every_n

        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)

        layer_windows = [
            None if is_global_layer(i, n_layer, global_every_n) else window_size
            for i in range(n_layer)
        ]
        print(f"Layer attention pattern (None = global/full-causal): {layer_windows}")

        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, n_kv_head, context_length, layer_windows[i], dropout)
            for i in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        if attention_mask is None and past_key_values is None:
            attention_mask = input_ids.ne(self.pad_id)

        x = self.drop(self.wte(input_ids))

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None
            if self.gradient_checkpointing and self.training and not use_cache:
                x = checkpoint.checkpoint(block, x, attention_mask, None, False, past_len,
                                          use_reentrant=False)
            else:
                out = block(x, attention_mask, past, use_cache, past_len)
                if use_cache:
                    x, present = out
                    presents.append(present)
                else:
                    x = out

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]
        running_mask = attention_mask

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)
            running_mask = torch.cat([running_mask, torch.ones_like(next_token)], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=running_mask,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

# MODEL V7

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE requires an even head_dim"
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_seq_len = max_seq_len
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)

class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.context_length = context_length
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

        k_exp = k.repeat_interleave(self.n_rep, dim=1)
        v_exp = v.repeat_interleave(self.n_rep, dim=1)

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if attn_mask is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)
        elif past_kv is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p)
        else:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        self.ln_1 = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, context_length, dropout)
        self.ln_2 = RMSNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        attn_out, present = self.attn(self.ln_1(x), attn_mask, past_kv, use_cache, position_offset)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return (x, present) if use_cache else x

class DecoderOnlyLM_RMSNorm_GQA(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head, n_kv_head,
                dropout=0.1, pad_id=0, gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.gradient_checkpointing = gradient_checkpointing
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, n_kv_head, context_length, dropout) for _ in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        combined_mask = None
        if attention_mask is not None:
            key_mask = attention_mask.bool()[:, None, None, :]
            if past_len == 0:
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
                eye = torch.eye(T, dtype=torch.bool, device=input_ids.device)
                combined_mask = (causal[None, None] & key_mask) | eye[None, None]
            else:
                combined_mask = key_mask

        x = self.drop(self.wte(input_ids))

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None
            if self.gradient_checkpointing and self.training and not use_cache:
                x = checkpoint.checkpoint(block, x, combined_mask, None, False, past_len,use_reentrant=False)
            else:
                out = block(x, combined_mask, past, use_cache, past_len)
                if use_cache:
                    x, present = out
                    presents.append(present)
                else:
                    x = out

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=attention_mask,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
def build_model():
    # for  v7
    return DecoderOnlyLM_RMSNorm_GQA(
    vocab_size=VOCAB_SIZE,         
    context_length=1536,
    n_embd=768, 
    n_layer=10,
    n_head=12, 
    n_kv_head=3,
    pad_id=PAD_ID,                 
    gradient_checkpointing=True,
) 
# for v2
#       return DecoderOnlyLM_RoPE(
#     vocab_size=VOCAB_SIZE,          
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     pad_id=PAD_ID,                  
#     gradient_checkpointing=True,
# )
# for v5
#       return DecoderOnlyLM_MoE_Sandwich(
#     vocab_size=VOCAB_SIZE,         
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     n_kv_head=2,
#     num_experts=2,
#     aux_loss_weight=0.01,
#     pad_id=PAD_ID,                  
#     gradient_checkpointing=True,
# )
# for v6
#     return DecoderOnlyLM_HybridMask(
#     vocab_size=VOCAB_SIZE,         
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     n_kv_head=2,
#     window_size=192,
#     global_every_n=4,
#     pad_id=PAD_ID,                  
#     gradient_checkpointing=True,
# )


HEAD_PATH   = "lm_head"      
LAYERS_PATH = "blocks"           
FINAL_NORM  = ("ln_f",)      
MODEL_PT    = "/kaggle/working/decoder_baseline_v7_final_model.pt"

In [ ]:
LABEL_KEYS = ("label", "labels", "y", "outcome", "judgment", "judgement")
 
def normalize_cases(raw):
    if isinstance(raw, dict):
        if isinstance(raw.get("data"), list):
            raw = raw["data"]
        else:
            raw = [{"id": k, **(v if isinstance(v, dict) else {"text": v})} for k, v in raw.items()]
    out = []
    for d in raw:
        text = d.get("text")
        if text is None:
            raise ValueError(f"case {d.get('id')!r} has no 'text' field; keys: {list(d)}")
        if isinstance(text, list):
            text = " ".join(str(x) for x in text)
        lab = next((d[k] for k in LABEL_KEYS if k in d), None)
        out.append({"id": str(d["id"]), "text": text, "label": int(lab) if lab is not None else None})
    return out
 
 
def cjpe_paths(root, include_test=True):
    p = {"train": f"{root}/Train/train.json", "dev": f"{root}/Train/dev.json"}
    if include_test:
        p["test"] = f"{root}/Test/test.json"
    return p
 
 
def download_cjpe_files(repo="Exploration-Lab/CS-781-Capstone", p4="Phase_4_CJPE", token=None, include_test=True):
    return {k: hf_hub_download(repo_id=repo, filename=f, repo_type="dataset", token=token)for k, f in cjpe_paths(p4, include_test).items()}
 
 
def tokenize_cases(cases, encode_fn, clean_fn=None, max_doc_tokens=None, desc="tokenizing"):
    out = []
    for c in tqdm(cases, desc=desc, leave=False):
        t = clean_fn(c["text"]) if clean_fn else c["text"]
        ids = np.asarray(encode_fn(t), dtype=np.int32)
        if max_doc_tokens and len(ids) > max_doc_tokens:
            ids = ids[-max_doc_tokens:]
        out.append({"id": c["id"], "label": c["label"], "ids": ids, "n_chars": len(c["text"])})
    return out
 
 
def setup_data(paths, tokenizer_pkl, sep_token="</s>", clean=True, clean_fn=None, max_doc_tokens=None,cache=None, include_test=True):
    if isinstance(paths, str):
        paths = cjpe_paths(paths, include_test)
    bpe_data, encode = load_tokenizer(tokenizer_pkl)
    stoi = bpe_data["stoi"]
    pad_id = stoi["<pad>"]
    print("special tokens:", [t for t in stoi if isinstance(t, str)])
    sep_id = stoi.get(sep_token) if sep_token else None
    if sep_token and sep_id is None:
        raise ValueError(f"{sep_token!r} not in the vocabulary; pass sep_token=<one of the special tokens above>")
    clean_fn = (clean_fn or clean_text) if clean else None
    splits = ["train", "dev"] + (["test"] if include_test and "test" in paths else [])
    key = {"max_doc_tokens": max_doc_tokens, "clean": bool(clean_fn), "splits": splits}
    D = {"pad_id": pad_id, "sep_id": sep_id, "vocab_size": bpe_data["vocab_size"], "encode": encode,"clean": clean_fn, "max_doc_tokens": max_doc_tokens, "bpe_data": bpe_data}
    cached = None
    if cache and os.path.exists(cache):
        cached = pickle.load(open(cache, "rb"))
        if cached.get("key") != key:
            print("cache settings differ -> re-tokenizing")
            cached = None
        else:
            print(f"loaded tokenized documents from {cache}")
    for split in splits:
        if cached:
            D[split] = cached[split]
        else:
            cases = normalize_cases(json.load(open(paths[split], encoding="utf-8")))
            D[split] = tokenize_cases(cases, encode, clean_fn, max_doc_tokens, desc=f"tokenizing {split}")
    if cache and not cached:
        pickle.dump({"key": key, **{s: D[s] for s in splits}}, open(cache, "wb"))
        print(f"saved tokenized documents to {cache}")
    for split in splits:
        docs = D[split]
        lens = np.array([len(d["ids"]) for d in docs])
        labs = Counter(d["label"] for d in docs)
        print(f"{split:5s}: {len(docs)} docs, labels {dict(labs)}, tokens/doc median {int(np.median(lens))} "
              f"p90 {int(np.percentile(lens, 90))} max {lens.max()}, docs over 1024 tokens: {100 * np.mean(lens > 1024):.0f}%")
    print(f"PAD_ID={pad_id}  VOCAB_SIZE={D['vocab_size']}  marker token id={sep_id}  cleaning={'on' if clean_fn else 'off'}")
    return D

In [ ]:
DOC_TAIL = dict(mode="tail", max_len=1024)
DOC_HEAD = dict(mode="head", max_len=1024)
DOC_HEADTAIL = dict(mode="headtail", max_len=1024, head_tokens=256)
DOC_CHUNKS = dict(mode="chunks", max_len=1024, stride=None, max_chunks=4, select="last", train_select="last")
 
 
def chunk_windows(n, size, stride):
    stride = max(1, min(stride, size))
    if n <= size:
        return [(0, n)]
    wins, s = [], n - size
    while s > 0:
        wins.append((s, s + size))
        s -= stride
    wins.append((0, size))
    return wins[::-1]
 
 
def select_windows(wins, max_chunks, select, rng=None):
    if not max_chunks or len(wins) <= max_chunks:
        return wins
    if select == "last":
        return wins[-max_chunks:]
    if select == "first":
        return wins[:max_chunks]
    if select == "even":
        idx = sorted(set(np.linspace(0, len(wins) - 1, max_chunks).round().astype(int).tolist()))
        return [wins[i] for i in idx]
    if select == "random":
        rng = rng or np.random.default_rng()
        idx = sorted(rng.choice(len(wins), size=max_chunks, replace=False).tolist())
        return [wins[i] for i in idx]
    raise ValueError(f"unknown select {select!r}")
 
 
def doc_to_chunks(ids, DOC, sep_id, training=False, rng=None):
    size = DOC["max_len"] - 1
    n, mode = len(ids), DOC["mode"]
    if mode == "tail":
        parts = [ids[-size:]]
    elif mode == "head":
        parts = [ids[:size]]
    elif mode == "headtail":
        if n <= size:
            parts = [ids]
        else:
            h = DOC.get("head_tokens", size // 4)
            parts = [np.concatenate([ids[:h], ids[-(size - h):]])]
    elif mode == "chunks":
        wins = chunk_windows(n, size, DOC.get("stride") or size)
        sel = DOC.get("train_select", DOC["select"]) if training else DOC["select"]
        parts = [ids[a:b] for a, b in select_windows(wins, DOC.get("max_chunks"), sel, rng)]
    else:
        raise ValueError(f"unknown mode {mode!r}")
    return [np.append(p, sep_id).astype(np.int64) for p in parts]
 
 
def collate_cjpe(batch, pad_id, DOC, sep_id, training=False, rng=None):
    chunks, doc_idx, chunk_pos, lengths = [], [], [], []
    for b_i, item in enumerate(batch):
        for k, c in enumerate(doc_to_chunks(item["ids"], DOC, sep_id, training, rng)):
            chunks.append(c); doc_idx.append(b_i); chunk_pos.append(k); lengths.append(len(c))
    ids = torch.full((len(chunks), max(lengths)), pad_id, dtype=torch.long)
    for i, c in enumerate(chunks):
        ids[i, :len(c)] = torch.from_numpy(c)
    labels = torch.tensor([-100 if it["label"] is None else it["label"] for it in batch], dtype=torch.long)
    return {"input_ids": ids, "lengths": torch.tensor(lengths), "doc_idx": torch.tensor(doc_idx),"chunk_pos": torch.tensor(chunk_pos), "n_docs": len(batch), "labels": labels}
 

In [ ]:
def load_phase1(build_fn, path):
    model = build_fn()
    ckpt = torch.load(path, map_location="cpu")
    sd = ckpt["model_state"] if isinstance(ckpt, dict) and "model_state" in ckpt else ckpt
    model.load_state_dict({k.replace("module.", "", 1): v for k, v in sd.items()}, strict=True)
    return model
 
 
class CJPEModel(nn.Module):
    def __init__(self, backbone, num_classes=2, head_path="lm_head", pool="last", agg="mean", dropout=0.1):
        super().__init__()
        assert pool in ("last", "mean") and agg in ("mean", "max", "attn", "last")
        self.backbone = backbone
        parent = backbone
        *parents, name = head_path.split(".")
        for p in parents:
            parent = getattr(parent, p)
        assert isinstance(getattr(parent, name), nn.Module), f"{head_path} not found"
        setattr(parent, name, nn.Identity())
        emb = next(m for m in backbone.modules() if isinstance(m, nn.Embedding))
        hidden = emb.embedding_dim
        self.vocab_size, self.num_classes, self.pool, self.agg = emb.num_embeddings, num_classes, pool, agg
        self.drop = nn.Dropout(dropout)
        self.att = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.Tanh(), nn.Linear(hidden // 2, 1)) if agg == "attn" else None
        self.classifier = nn.Linear(hidden, num_classes)
 
    def head_parameters(self):
        ps = list(self.classifier.parameters())
        if self.att is not None:
            ps += list(self.att.parameters())
        return ps
 
    def encode_chunks(self, input_ids, lengths):
        out = self.backbone(input_ids)
        h = out["logits"] if isinstance(out, dict) else out                       
        if self.pool == "last":
            idx = (lengths - 1).view(-1, 1, 1).expand(-1, 1, h.size(-1))
            return h.gather(1, idx).squeeze(1)
        m = (torch.arange(h.size(1), device=h.device)[None, :] < lengths[:, None]).unsqueeze(-1).to(h.dtype)
        return (h * m).sum(1) / m.sum(1).clamp(min=1)
 
    def forward(self, input_ids, lengths, doc_idx, chunk_pos, n_docs):
        v = self.encode_chunks(input_ids, lengths)                                
        Cmax = int(chunk_pos.max().item()) + 1
        pad = v.new_zeros(n_docs, Cmax, v.size(-1)).index_put((doc_idx, chunk_pos), v)
        mask = torch.zeros(n_docs, Cmax, dtype=torch.bool, device=v.device).index_put(
            (doc_idx, chunk_pos), torch.ones_like(doc_idx, dtype=torch.bool))
        mf = mask.unsqueeze(-1).to(pad.dtype)
        if self.agg == "mean":
            d = (pad * mf).sum(1) / mf.sum(1).clamp(min=1)
        elif self.agg == "max":
            d = pad.masked_fill(~mask.unsqueeze(-1), float("-inf")).max(1).values
        elif self.agg == "attn":
            s = self.att(pad).squeeze(-1).float().masked_fill(~mask, float("-inf"))
            d = (torch.softmax(s, dim=1).unsqueeze(-1).to(pad.dtype) * pad).sum(1)
        else:                                                                     
            d = pad[torch.arange(n_docs, device=pad.device), mask.sum(1) - 1]
        return self.classifier(self.drop(d))                                      
 
def trainable_state(model):
    return {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
 
def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot

In [ ]:
def binary_scores(gold, pred):
    gold, pred = np.asarray(gold), np.asarray(pred)
    per = {}
    for k in (0, 1):
        tp = int(np.sum((gold == k) & (pred == k)))
        fp = int(np.sum((gold != k) & (pred == k)))
        fn = int(np.sum((gold == k) & (pred != k)))
        P = tp / (tp + fp) if tp + fp else 0.0
        R = tp / (tp + fn) if tp + fn else 0.0
        per[k] = {"precision": P, "recall": R, "f1": 2 * P * R / (P + R) if P + R else 0.0, "support": tp + fn}
    macro = {m: float(np.mean([per[0][m], per[1][m]])) for m in ("precision", "recall", "f1")}
    conf = {"tn": int(np.sum((gold == 0) & (pred == 0))), "fp": int(np.sum((gold == 0) & (pred == 1))),
            "fn": int(np.sum((gold == 1) & (pred == 0))), "tp": int(np.sum((gold == 1) & (pred == 1)))}
    return {"per_class": per, "macro": macro, "accuracy": float(np.mean(gold == pred)), "confusion": conf}
 
 
@torch.no_grad()
def predict_cjpe(model, docs, DOC, pad_id, sep_id, device, batch_size=2, use_amp=True):
    model.eval()
    amp = use_amp and device.type == "cuda"
    order = sorted(range(len(docs)), key=lambda i: len(docs[i]["ids"]))
    probs = np.zeros(len(docs), dtype=np.float32)
    for s in range(0, len(order), batch_size):
        idxs = order[s:s + batch_size]
        b = collate_cjpe([docs[i] for i in idxs], pad_id, DOC, sep_id, training=False)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            logits = model(b["input_ids"].to(device), b["lengths"].to(device), b["doc_idx"].to(device),
                           b["chunk_pos"].to(device), b["n_docs"])
        probs[idxs] = torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy()
    return probs
 
 
def evaluate_cjpe(model, docs, DOC, pad_id, sep_id, device, threshold=0.5, **kw):
    probs = predict_cjpe(model, docs, DOC, pad_id, sep_id, device, **kw)
    gold = [d["label"] for d in docs]
    return binary_scores(gold, (probs >= threshold).astype(int)), probs
 
 
def tune_threshold(probs, gold, grid=None):
    grid = grid if grid is not None else np.linspace(0.05, 0.95, 91)
    best_t = max(grid, key=lambda t: binary_scores(gold, (probs >= t).astype(int))["macro"]["f1"])
    return float(best_t), binary_scores(gold, (probs >= best_t).astype(int))
 
 
def print_report(scores, name=""):
    m, c = scores["macro"], scores["confusion"]
    print(f"{name} macro-F1 {m['f1']:.4f} (P {m['precision']:.4f}, R {m['recall']:.4f}) acc {scores['accuracy']:.4f} | "
          f"class0 F1 {scores['per_class'][0]['f1']:.3f} (n={scores['per_class'][0]['support']}), "
          f"class1 F1 {scores['per_class'][1]['f1']:.3f} (n={scores['per_class'][1]['support']}) | "
          f"TN {c['tn']} FP {c['fp']} FN {c['fn']} TP {c['tp']}")
 

In [ ]:
CFG_CJPE = dict(epochs=5, batch_size=2, grad_accum=8, lr_backbone=2e-5, lr_head=1e-3, weight_decay=0.01,warmup_ratio=0.1, max_grad_norm=1.0, fp16=True, class_weight=None, seed=0)   # class_weight: None|'sqrt'|'inv'
 
def class_weights(docs, mode):
    if not mode:
        return None
    cnt = np.bincount([d["label"] for d in docs], minlength=2).astype(float) + 1.0
    w = cnt.sum() / (2 * cnt)
    if mode == "sqrt":
        w = np.sqrt(w)
    return torch.tensor(w / w.mean(), dtype=torch.float)
 
 
def train_cjpe(model, train_docs, pad_id, sep_id, DOC, cfg, device, val_fn=None, save_path=None, class_w=None,extra_loss=None, on_step=None):
    model.to(device)
    rng = np.random.default_rng(cfg.get("seed", 0))
    head_ids = {id(p) for p in model.head_parameters()}
    head_params = [p for p in model.parameters() if id(p) in head_ids and p.requires_grad]
    other = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    groups = [{"params": head_params, "lr": cfg["lr_head"]}]
    if other:
        groups.append({"params": other, "lr": cfg["lr_backbone"]})
    opt = torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])
 
    loader = DataLoader(train_docs, batch_size=cfg["batch_size"], shuffle=True,
                        collate_fn=lambda b: collate_cjpe(b, pad_id, DOC, sep_id, training=True, rng=rng))
    total_steps = max(1, (len(loader) // cfg["grad_accum"]) * cfg["epochs"])
    warmup = int(cfg["warmup_ratio"] * total_steps)
 
    def lr_lambda(step):
        if step < warmup:
            return step / max(1, warmup)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
 
    use_amp = cfg["fp16"] and device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    cw = class_w.to(device) if class_w is not None else None
    best, history, step = -1.0, [], 0
    for epoch in range(cfg["epochs"]):
        model.train()
        run, nb, t0 = 0.0, 0, time.time()
        opt.zero_grad()
        pbar = tqdm(loader, desc=f"epoch {epoch + 1}/{cfg['epochs']}", leave=False)
        for i, b in enumerate(pbar):
            ids, ln, di, cp = (b[k].to(device) for k in ("input_ids", "lengths", "doc_idx", "chunk_pos"))
            y = b["labels"].to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(ids, ln, di, cp, b["n_docs"])
            loss = F.cross_entropy(logits.float(), y, weight=cw)
            if extra_loss is not None:
                loss = loss + extra_loss(model)
            scaler.scale(loss / cfg["grad_accum"]).backward()
            run += loss.item(); nb += 1
            pbar.set_postfix(loss=f"{run / nb:.4f}")
            if (i + 1) % cfg["grad_accum"] == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
                scaler.step(opt); scaler.update(); sched.step()
                step += 1
                if on_step is not None:
                    on_step(model, step)
                opt.zero_grad()
        rec = {"epoch": epoch + 1, "train_loss": run / max(1, nb), "sec": round(time.time() - t0)}
        if val_fn is not None:
            rec["val"] = val_fn(model)
            if rec["val"] > best:
                best = rec["val"]
                if save_path:
                    torch.save(trainable_state(model), save_path)
        history.append(rec)
        print(rec)
    if val_fn is not None and save_path:
        model.load_state_dict(torch.load(save_path, map_location=device), strict=False)
        print(f"restored best epoch (val macro-F1={best:.4f})")
    return history

In [ ]:
def predict_cases(model, raw_cases, D, DOC, device, threshold=0.5, batch_size=2):
    cases = normalize_cases(raw_cases)
    docs = tokenize_cases(cases, D["encode"], D["clean"], D["max_doc_tokens"], desc="tokenizing cases")
    probs = predict_cjpe(model, docs, DOC, D["pad_id"], D["sep_id"], device, batch_size=batch_size)
    preds = [{"id": c["id"], "pred": int(p >= threshold)} for c, p in zip(cases, probs)]
    return preds
 
 
def write_cjpe_submission(preds, out_dir="/kaggle/working", zip_name="submission_cjpe.zip"):
    path = os.path.join(out_dir, "prediction.json")
    json.dump(preds, open(path, "w", encoding="utf-8"))
    zp = os.path.join(out_dir, zip_name)
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    print(f"wrote {zp}: {len(preds)} predictions, {sum(p['pred'] for p in preds)} predicted as 1")

In [ ]:
def run_full_ft_cjpe(build_fn, model_pt, head_path, D, DOC, device, tag="cjpe_full", pool="last", agg="mean",cfg=None, train_subset=None, eval_batch_size=2):
    cfg = cfg or CFG_CJPE
    model = CJPEModel(load_phase1(build_fn, model_pt), 2, head_path=head_path, pool=pool, agg=agg)
    for p in model.parameters():
        p.requires_grad = True
    count_params(model)
    train_docs = D["train"]
    if train_subset:
        pick = np.random.default_rng(cfg.get("seed", 0)).choice(len(train_docs), size=min(train_subset, len(train_docs)), replace=False)
        train_docs = [train_docs[i] for i in sorted(pick)]
    print(f"training on {len(train_docs)} documents; document representation: {DOC}; pool={pool}, agg={agg}")
 
    def val_fn(m):
        scores, _ = evaluate_cjpe(m, D["dev"], DOC, D["pad_id"], D["sep_id"], device, batch_size=eval_batch_size)
        print_report(scores, "   dev:")
        return scores["macro"]["f1"]
 
    cw = class_weights(train_docs, cfg.get("class_weight"))
    hist = train_cjpe(model, train_docs, D["pad_id"], D["sep_id"], DOC, cfg, device, val_fn=val_fn,save_path=f"/kaggle/working/{tag}_best.pt", class_w=cw)
    return model, hist

In [ ]:
def adalora_budget(step, total, tinit, tfinal, init_bgt, target_bgt):
    if step <= tinit:
        return init_bgt
    if step > total - tfinal:
        return target_bgt
    mul = 1 - (step - tinit) / (total - tfinal - tinit)
    return int((init_bgt - target_bgt) * mul ** 3 + target_bgt)


class AdaLoRALinear(nn.Module):
    def __init__(self, base, r=12, alpha=24, dropout=0.0):
        super().__init__()
        self.base = base                                          
        for p in self.base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features, self.r = base.in_features, base.out_features, r
        self.scale = alpha / r                                     
        dev = base.weight.device
        self.lora_A = nn.Parameter(torch.empty(r, self.in_features, device=dev))     
        self.lora_E = nn.Parameter(torch.zeros(r, 1, device=dev))                    
        self.lora_B = nn.Parameter(torch.empty(self.out_features, r, device=dev))    
        nn.init.normal_(self.lora_A, mean=0.0, std=0.02)
        nn.init.normal_(self.lora_B, mean=0.0, std=0.02)
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.lora_A * self.lora_E), self.lora_B)

    @torch.no_grad()
    def merged_weight(self):
        return self.base.weight + self.scale * (self.lora_B @ (self.lora_A * self.lora_E))


def apply_adalora(model, init_r=12, alpha=24, dropout=0.1, targets=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc")):
    for p in model.parameters():
        p.requires_grad = False
    targets, n = set(targets), 0
    for _, parent in list(model.backbone.named_modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and cname in targets:
                setattr(parent, cname, AdaLoRALinear(child, init_r, alpha, dropout))
                n += 1
    assert n > 0, f"no nn.Linear layers matched {targets}"
    for p in model.head_parameters():
        p.requires_grad = True
    print(f"AdaLoRA: wrapped {n} Linear layers with initial rank {init_r}")
    count_params(model)
    return model


def orth_regularizer(model):
    total, n = 0.0, 0
    for m in model.modules():
        if isinstance(m, AdaLoRALinear):
            I = torch.eye(m.r, device=m.lora_A.device)
            total = total + torch.norm(m.lora_A @ m.lora_A.T - I, p="fro") + torch.norm(m.lora_B.T @ m.lora_B - I, p="fro")
            n += 2
    return total / max(1, n)


class AdaLoRAController:
    def __init__(self, model, init_r, target_r, total_steps, tinit=0.1, tfinal=0.25, delta_t=10, beta1=0.85, beta2=0.85):
        self.layers = [m for m in model.modules() if isinstance(m, AdaLoRALinear)]
        self.init_bgt = sum(m.r for m in self.layers)
        self.target_bgt = len(self.layers) * target_r
        self.T = total_steps
        self.tinit = int(tinit * total_steps) if tinit < 1 else int(tinit)          
        self.tfinal = int(tfinal * total_steps) if tfinal < 1 else int(tfinal)
        self.delta_t = max(1, min(delta_t, self.T // 20))                            
        self.b1, self.b2 = beta1, beta2
        self.masks = None
        self.reset_ipt()
        print(f"AdaLoRA budget: {self.init_bgt} -> {self.target_bgt} triplets over {self.T} steps "f"(warm-up {self.tinit}, final fine-tuning {self.tfinal}, re-allocate every {self.delta_t} steps)")

    def reset_ipt(self):
        self.state = {}

    @torch.no_grad()
    def update_ipt(self):
        params = [(l, n, getattr(l, f"lora_{n}")) for l in self.layers for n in ("A", "B", "E")]
        grads = [p.grad for _, _, p in params]
        if any(g is None for g in grads):
            return
        if torch.stack([~torch.isfinite(g).all() for g in grads]).any().item():   
            return
        for (l, n, p), g in zip(params, grads):
            st = self.state.setdefault((id(l), n), {"avg": torch.zeros_like(p), "unc": torch.zeros_like(p)})
            ipt = (p * g).abs()
            st["avg"].mul_(self.b1).add_(ipt, alpha=1 - self.b1)
            st["unc"].mul_(self.b2).add_((ipt - st["avg"]).abs(), alpha=1 - self.b2)

    def _scores(self):
        out = []
        for l in self.layers:
            sA, sB, sE = (self.state[(id(l), n)] for n in ("A", "B", "E"))
            e = (sE["avg"] * sE["unc"]).view(-1)                                   
            a = (sA["avg"] * sA["unc"]).mean(dim=1)                                 
            b = (sB["avg"] * sB["unc"]).mean(dim=0)                                 
            out.append(e + a + b)
        return out

    @torch.no_grad()
    def mask_to_budget(self, budget):
        if not self.state:
            return
        scores = self._scores()
        k = int(self.init_bgt - budget)                                              
        if k <= 0:
            self.masks = None
            return
        thr = torch.kthvalue(torch.cat(scores), k).values.item()
        self.masks = [s <= thr for s in scores]
        self._apply_masks()

    @torch.no_grad()
    def _apply_masks(self):
        for l, m in zip(self.layers, self.masks):
            l.lora_E.data[m] = 0.0

    def on_step(self, model, step):
        T, ti, tf = self.T, self.tinit, self.tfinal
        if step < T - tf:
            self.update_ipt()
            if step > ti and step % self.delta_t == 0:
                self.mask_to_budget(adalora_budget(step, T, ti, tf, self.init_bgt, self.target_bgt))
        elif step == T - tf:                                                        
            self.mask_to_budget(self.target_bgt)
            self.reset_ipt()
        elif self.masks is not None:                                                 
            self._apply_masks()


def rank_report(model):
    rows = [(n, int((m.lora_E.detach() != 0).sum().item()), m.r) for n, m in model.named_modules() if isinstance(m, AdaLoRALinear)]
    act, tot = sum(a for _, a, _ in rows), sum(r for _, _, r in rows)
    print(f"active rank triplets: {act}/{tot} ({100 * act / tot:.0f}%), average rank per matrix {act / len(rows):.2f}")
    by_type, by_block = {}, {}
    for n, a, r in rows:
        by_type.setdefault(n.split(".")[-2] + "." + n.split(".")[-1] if n.count(".") >= 1 else n, []).append(a)
        mb = re.search(r"blocks\.(\d+)\.", n)
        if mb:
            by_block.setdefault(int(mb.group(1)), []).append(a)
    print("  mean rank by matrix:", {k: round(sum(v) / len(v), 2) for k, v in sorted(by_type.items())})
    print("  mean rank by block :", {k: round(sum(v) / len(v), 2) for k, v in sorted(by_block.items())})
    return rows


def effective_params(model):
    n = sum(int((m.lora_E.detach() != 0).sum().item()) * (m.in_features + m.out_features + 1)
            for m in model.modules() if isinstance(m, AdaLoRALinear))
    head = sum(p.numel() for p in model.head_parameters())
    print(f"effective AdaLoRA parameters: {n/1e6:.3f}M (+ head {head/1e6:.3f}M)")
    return n + head


@torch.no_grad()
def merge_adalora(model):
    for _, parent in list(model.backbone.named_modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, AdaLoRALinear):
                lin = nn.Linear(child.in_features, child.out_features, bias=child.base.bias is not None).to(child.base.weight.device)
                lin.weight.copy_(child.merged_weight())
                if child.base.bias is not None:
                    lin.bias.copy_(child.base.bias)
                for q in lin.parameters():
                    q.requires_grad = False
                setattr(parent, cname, lin)
    return model


@torch.no_grad()
def check_adalora_identity(build_fn, model_pt, head_path, device, pool="last", agg="mean", **kw):
    model = CJPEModel(load_phase1(build_fn, model_pt), 2, head_path=head_path, pool=pool, agg=agg).to(device).eval()
    batch = dummy_cjpe_batch(model.vocab_size, device)
    ref = model(*batch)
    apply_adalora(model, **kw)
    model.to(device).eval()
    print("max |diff| vs un-adapted model at init:", (model(*batch) - ref).abs().max().item(), "(expect 0)")


CFG_ADALORA = {**CFG_CJPE, "lr_backbone": 2e-5}                                    

def run_adalora_cjpe(build_fn, model_pt, head_path, D, DOC, device, tag="cjpe_adalora", pool="last", agg="mean",
                     init_r=12, target_r=8, alpha=24, dropout=0.1, orth_weight=0.1,
                     targets=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc"),
                     tinit=0.1, tfinal=0.25, delta_t=10, beta1=0.85, beta2=0.85,
                     cfg=None, train_subset=None, eval_batch_size=2):

    cfg = cfg or CFG_ADALORA
    model = CJPEModel(load_phase1(build_fn, model_pt), 2, head_path=head_path, pool=pool, agg=agg)
    model = apply_adalora(model, init_r, alpha, dropout, targets)
    train_docs = D["train"]
    if train_subset:
        pick = np.random.default_rng(cfg.get("seed", 0)).choice(len(train_docs), size=min(train_subset, len(train_docs)), replace=False)
        train_docs = [train_docs[i] for i in sorted(pick)]
    total_steps = max(1, (math.ceil(len(train_docs) / cfg["batch_size"]) // cfg["grad_accum"]) * cfg["epochs"])
    ctrl = AdaLoRAController(model, init_r, target_r, total_steps, tinit, tfinal, delta_t, beta1, beta2)
    print(f"training on {len(train_docs)} documents; document representation: {DOC}; pool={pool}, agg={agg}")

    def val_fn(m):
        scores, _ = evaluate_cjpe(m, D["dev"], DOC, D["pad_id"], D["sep_id"], device, batch_size=eval_batch_size)
        print_report(scores, "   dev:")
        return scores["macro"]["f1"]

    cw = class_weights(train_docs, cfg.get("class_weight"))
    hist = train_cjpe(model, train_docs, D["pad_id"], D["sep_id"], DOC, cfg, device, val_fn=val_fn,
                      save_path=f"/kaggle/working/{tag}_best.pt", class_w=cw,
                      extra_loss=lambda m: orth_weight * orth_regularizer(m), on_step=ctrl.on_step)
    rank_report(model)
    effective_params(model)
    return model, hist

In [ ]:
_ROT_IDX = {}

def _rot_index(d, device):
    key = (d, str(device))
    if key not in _ROT_IDX:
        i = torch.arange(d, device=device).unsqueeze(1)
        j = torch.arange(d, device=device).unsqueeze(0)
        _ROT_IDX[key] = (j - i) % d                     
    return _ROT_IDX[key]


class LoRMALinear(nn.Module):
    def __init__(self, base, r=32, alpha=64, variant="plus", dropout=0.1):
        super().__init__()
        assert variant in ("plus", "pi")
        self.in_features, self.out_features = base.in_features, base.out_features
        self.weight, self.bias = base.weight, base.bias            
        self.weight.requires_grad = False
        if self.bias is not None:
            self.bias.requires_grad = False
        d, dev = base.out_features, base.weight.device
        self.variant, self.r, self.scale = variant, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, d, device=dev))
        self.B = nn.Parameter(torch.empty(d, r, device=dev))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        with torch.no_grad():
            if variant == "plus":
                nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
                self.B.zero_()
            else:
                self.A.zero_()
                self.A[0, 0] = 1.0
                nn.init.kaiming_uniform_(self.B, a=math.sqrt(5))
                self.B[:, 0] = 1.0

    def forward(self, x):
        y = F.linear(x, self.weight)                               
        if self.variant == "plus":                                
            y = y + self.scale * F.linear(F.linear(self.drop(y), self.A), self.B)
        else:                                                      
            M = torch.gather(self.B @ self.A, 1, _rot_index(self.out_features, y.device))
            y = F.linear(y, M)
        return y if self.bias is None else y + self.bias

    @torch.no_grad()
    def merged_weight(self):
        d = self.out_features
        if self.variant == "plus":
            M = torch.eye(d, device=self.weight.device) + self.scale * (self.B @ self.A)
        else:
            M = torch.gather(self.B @ self.A, 1, _rot_index(d, self.weight.device))
        return M @ self.weight


def apply_lorma(model, r=32, alpha=64, variant="plus", dropout=0.0, targets=("q_proj", "k_proj", "v_proj", "c_proj")):
    for p in model.parameters():
        p.requires_grad = False
    targets, n = set(targets), 0
    for _, parent in list(model.backbone.named_modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and cname in targets:
                setattr(parent, cname, LoRMALinear(child, r, alpha, variant, dropout))
                n += 1
    for p in model.head_parameters():
        p.requires_grad = True
    print(f"LoRMA-{variant}: adapted {n} Linear layers (r={r})")
    count_params(model)
    return model


@torch.no_grad()
def merge_lorma(model):
    for _, parent in list(model.backbone.named_modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, LoRMALinear):
                lin = nn.Linear(child.in_features, child.out_features, bias=child.bias is not None).to(child.weight.device)
                lin.weight.copy_(child.merged_weight())
                if child.bias is not None:
                    lin.bias.copy_(child.bias)
                for q in lin.parameters():
                    q.requires_grad = False
                setattr(parent, cname, lin)
    return model


@torch.no_grad()
def check_lorma_identity(build_fn, model_pt, head_path, device, variant="plus", pool="last", agg="mean", **kw):
    model = CJPEModel(load_phase1(build_fn, model_pt), 2, head_path=head_path, pool=pool, agg=agg).to(device).eval()
    batch = dummy_cjpe_batch(model.vocab_size, device)
    ref = model(*batch)
    apply_lorma(model, variant=variant, **kw)
    model.to(device).eval()
    print(f"{variant}: max |diff| vs un-adapted model at init:", (model(*batch) - ref).abs().max().item())


CFG_LORMA = {**CFG_CJPE, "lr_backbone": 5e-4}                    


def run_lorma_cjpe(build_fn, model_pt, head_path, D, DOC, device, tag="cjpe_lorma", pool="last", agg="mean",
                   variant="plus", r=32, alpha=64, dropout=0.1, targets=("q_proj", "k_proj", "v_proj", "c_proj"),
                   cfg=None, train_subset=None, eval_batch_size=2):

    cfg = cfg or CFG_LORMA
    model = CJPEModel(load_phase1(build_fn, model_pt), 2, head_path=head_path, pool=pool, agg=agg)
    model = apply_lorma(model, r, alpha, variant, dropout, targets)
    train_docs = D["train"]
    if train_subset:
        pick = np.random.default_rng(cfg.get("seed", 0)).choice(len(train_docs), size=min(train_subset, len(train_docs)), replace=False)
        train_docs = [train_docs[i] for i in sorted(pick)]
    print(f"training on {len(train_docs)} documents; document representation: {DOC}; pool={pool}, agg={agg}")

    def val_fn(m):
        scores, _ = evaluate_cjpe(m, D["dev"], DOC, D["pad_id"], D["sep_id"], device, batch_size=eval_batch_size)
        print_report(scores, "   dev:")
        return scores["macro"]["f1"]

    cw = class_weights(train_docs, cfg.get("class_weight"))
    hist = train_cjpe(model, train_docs, D["pad_id"], D["sep_id"], DOC, cfg, device, val_fn=val_fn,
                      save_path=f"/kaggle/working/{tag}_best.pt", class_w=cw)
    return model, hist

In [ ]:
def student_by_truncation(build_fn, model_pt, head_path, keep_layers=5, pool="last", agg="mean", blocks_attr="blocks"):
    bb = load_phase1(build_fn, model_pt)
    blocks = getattr(bb, blocks_attr)
    n = len(blocks)
    idx = sorted(set(int(round(i)) for i in np.linspace(0, n - 1, keep_layers)))
    setattr(bb, blocks_attr, nn.ModuleList([blocks[i] for i in idx]))
    print(f"student keeps blocks {idx} of {n}")
    return CJPEModel(bb, 2, head_path=head_path, pool=pool, agg=agg)


def student_from_model(student_build_fn, student_pt, head_path, pool="last", agg="mean"):
    return CJPEModel(load_phase1(student_build_fn, student_pt), 2, head_path=head_path, pool=pool, agg=agg)


def rebuild_teacher(build_fn, model_pt, head_path, ckpt_path, device, adapt=None, pool="last", agg="mean"):
    model = CJPEModel(load_phase1(build_fn, model_pt), 2, head_path=head_path, pool=pool, agg=agg)
    if adapt is None:
        for p in model.parameters():
            p.requires_grad = True
    else:
        model = adapt(model)
    sd = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(sd, strict=False)
    trainable = {n for n, p in model.named_parameters() if p.requires_grad}
    print(f"loaded {len(sd)} tensors; trainable params missing from the checkpoint: {len(trainable - set(sd))}")
    return model.to(device).eval()

@torch.no_grad()
def logp_cjpe(model, docs, DOC, pad_id, sep_id, device, batch_size=2, use_amp=True, desc="teacher"):
    model.eval()
    amp = use_amp and device.type == "cuda"
    order = sorted(range(len(docs)), key=lambda i: len(docs[i]["ids"]))
    out = np.zeros((len(docs), 2), dtype=np.float32)
    for s in tqdm(range(0, len(order), batch_size), desc=desc, leave=False):
        idxs = order[s:s + batch_size]
        b = collate_cjpe([docs[i] for i in idxs], pad_id, DOC, sep_id, training=False)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            logits = model(b["input_ids"].to(device), b["lengths"].to(device), b["doc_idx"].to(device),
                           b["chunk_pos"].to(device), b["n_docs"])
        out[idxs] = torch.log_softmax(logits.float(), dim=-1).cpu().numpy()
    return out


def teacher_logp(teachers, D, device, split="train", batch_size=2):
    acc = 0
    for t in teachers:
        model, DOC, *rest = t
        Dt = rest[0] if rest else D
        acc = acc + logp_cjpe(model, Dt[split], DOC, Dt["pad_id"], Dt["sep_id"], device, batch_size)
    return acc / len(teachers)


def attach_soft(docs, soft, teacher_ids=None):
    assert len(docs) == len(soft), f"{len(docs)} documents but {len(soft)} teacher outputs"
    if teacher_ids is not None:
        assert [d["id"] for d in docs] == list(teacher_ids), "student and teacher documents are in a different order"
    return [{**d, "soft": soft[i]} for i, d in enumerate(docs)]

def collate_kd(batch, pad_id, DOC, sep_id, training=False, rng=None):
    b = collate_cjpe(batch, pad_id, DOC, sep_id, training, rng)
    b["soft"] = torch.from_numpy(np.stack([it["soft"] for it in batch]).astype(np.float32))
    return b


def kd_loss(logits, y, soft_logp, T, kd_alpha, class_w=None):
    ce = F.cross_entropy(logits, y, weight=class_w)
    s_logp = F.log_softmax(logits / T, dim=-1)
    t_prob = F.softmax(soft_logp / T, dim=-1)                          
    kd = (t_prob * (torch.log(t_prob.clamp_min(1e-9)) - s_logp)).sum(-1).mean() * (T * T)
    return kd_alpha * ce + (1 - kd_alpha) * kd, ce.detach(), kd.detach()


def train_cjpe_kd(model, docs, pad_id, sep_id, DOC, cfg, device, val_fn=None, save_path=None, class_w=None,T=2.0, kd_alpha=0.5):
    model.to(device)
    rng = np.random.default_rng(cfg.get("seed", 0))
    head_ids = {id(p) for p in model.head_parameters()}
    head_params = [p for p in model.parameters() if id(p) in head_ids and p.requires_grad]
    other = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    groups = [{"params": head_params, "lr": cfg["lr_head"]}]
    if other:
        groups.append({"params": other, "lr": cfg["lr_backbone"]})
    opt = torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])

    loader = DataLoader(docs, batch_size=cfg["batch_size"], shuffle=True,
                        collate_fn=lambda b: collate_kd(b, pad_id, DOC, sep_id, training=True, rng=rng))
    total_steps = max(1, (len(loader) // cfg["grad_accum"]) * cfg["epochs"])
    warmup = int(cfg["warmup_ratio"] * total_steps)

    def lr_lambda(step):
        if step < warmup:
            return step / max(1, warmup)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    use_amp = cfg["fp16"] and device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    cw = class_w.to(device) if class_w is not None else None
    best, history = -1.0, []
    for epoch in range(cfg["epochs"]):
        model.train()
        run, run_ce, run_kd, nb, t0 = 0.0, 0.0, 0.0, 0, time.time()
        opt.zero_grad()
        pbar = tqdm(loader, desc=f"epoch {epoch + 1}/{cfg['epochs']}", leave=False)
        for i, b in enumerate(pbar):
            ids, ln, di, cp = (b[k].to(device) for k in ("input_ids", "lengths", "doc_idx", "chunk_pos"))
            y, soft = b["labels"].to(device), b["soft"].to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(ids, ln, di, cp, b["n_docs"])
            loss, ce, kd = kd_loss(logits.float(), y, soft, T, kd_alpha, cw)
            scaler.scale(loss / cfg["grad_accum"]).backward()
            run += loss.item(); run_ce += ce.item(); run_kd += kd.item(); nb += 1
            pbar.set_postfix(loss=f"{run / nb:.4f}")
            if (i + 1) % cfg["grad_accum"] == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
        nb = max(1, nb)
        rec = {"epoch": epoch + 1, "train_loss": run / nb, "ce": run_ce / nb, "kd": run_kd / nb, "sec": round(time.time() - t0)}
        if val_fn is not None:
            rec["val"] = val_fn(model)
            if rec["val"] > best:
                best = rec["val"]
                if save_path:
                    torch.save(trainable_state(model), save_path)
        history.append(rec)
        print(rec)
    if val_fn is not None and save_path:
        model.load_state_dict(torch.load(save_path, map_location=device), strict=False)
        print(f"restored best epoch (val macro-F1={best:.4f})")
    return history

CFG_KD = {**CFG_CJPE, "lr_backbone": 5e-4, "lr_head": 1e-3}          

def run_kd_cjpe(teachers, student, D, DOC, device, tag="cjpe_kd", T=2.0, kd_alpha=0.5, D_student=None,cfg=None, train_subset=None, eval_batch_size=2):
    cfg = cfg or CFG_KD
    Ds = D_student or D
    ctx = getattr(student.backbone, "context_length", None)
    assert ctx is None or ctx >= DOC["max_len"], "student context length < DOC['max_len']"
    soft = teacher_logp(teachers, D, device, "train")
    gold = np.array([d["label"] for d in D["train"]])
    conf = float(np.exp(soft.max(1)).mean())
    print(f"teacher on the training documents: accuracy {np.mean(soft.argmax(1) == gold):.3f},"f"mean confidence {conf:.3f}  (very sharp targets give the student little beyond the gold labels: raise T)")
    docs = attach_soft(Ds["train"], soft, [d["id"] for d in D["train"]])
    if train_subset:
        pick = np.random.default_rng(cfg.get("seed", 0)).choice(len(docs), size=min(train_subset, len(docs)), replace=False)
        docs = [docs[i] for i in sorted(pick)]
    for p in student.parameters():
        p.requires_grad = True
    count_params(student)
    print(f"KD on {len(docs)} documents, T={T}, kd_alpha={kd_alpha}; student document representation: {DOC}")

    def val_fn(m):
        scores, _ = evaluate_cjpe(m, Ds["dev"], DOC, Ds["pad_id"], Ds["sep_id"], device, batch_size=eval_batch_size)
        print_report(scores, "   dev:")
        return scores["macro"]["f1"]

    cw = class_weights(docs, cfg.get("class_weight"))
    hist = train_cjpe_kd(student, docs, Ds["pad_id"], Ds["sep_id"], DOC, cfg, device, val_fn=val_fn,save_path=f"/kaggle/working/{tag}_best.pt", class_w=cw, T=T, kd_alpha=kd_alpha)
    return student, hist


def kd_report(models, device, n_docs=None):
    for name, (m, DOC, Dm) in models.items():
        docs = Dm["dev"][:n_docs] if n_docs else Dm["dev"]
        n_par = sum(p.numel() for p in m.parameters())
        mb = sum(p.numel() * p.element_size() for p in m.parameters()) / 1e6
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        probs = predict_cjpe(m, docs, DOC, Dm["pad_id"], Dm["sep_id"], device)
        if device.type == "cuda":
            torch.cuda.synchronize()
        sec = time.time() - t0
        gold = [d["label"] for d in docs]
        f05 = binary_scores(gold, (probs >= 0.5).astype(int))["macro"]["f1"]
        thr, sc = tune_threshold(probs, gold)
        print(f"{name:10s} params {n_par/1e6:7.2f}M ({mb:6.0f} MB) | dev macro-F1 {f05:.4f} @0.5, {sc['macro']['f1']:.4f} @{thr:.2f} "f"| dev inference {sec:.0f}s")

In [ ]:
P4 = "Phase_4_CJPE"
p4 = download_cjpe_files(hf_dataset_repo, P4, HF_TOKEN,include_test=True)                 
D = setup_data(p4, "/kaggle/working/legal_bpe_tokenizer.pkl", cache="/kaggle/working/cjpe_tok.pkl")
PAD_ID, VOCAB_SIZE = D["pad_id"], D["vocab_size"]

In [ ]:
model_ft, hist_ft = run_full_ft_cjpe(build_model, MODEL_PT, HEAD_PATH, D, DOC_TAIL, device, tag="cjpe_ft",cfg=CFG_CJPE)
model_adalora, hist_adalora = run_adalora_cjpe(build_model, MODEL_PT, HEAD_PATH, D, DOC_TAIL, device, tag="cjpe_adalora")
model_lorma_plus, hist_lorma_plus = run_lorma_cjpe(build_model, MODEL_PT, HEAD_PATH, D, DOC_TAIL, device, tag="cjpe_lorma_plus", variant="plus")

DOC_TEACHER = DOC_TAIL       

student = student_by_truncation(build_model, MODEL_PT, HEAD_PATH, keep_layers=5)
student, hist_kd = run_kd_cjpe([(model_adalora, DOC_TEACHER)], student, D, DOC_TAIL, device, tag="cjpe_kd_ada")
kd_report({"teacher (AdaLoRA)": (model_adalora, DOC_TEACHER, D), "student": (student, DOC_TAIL, D)}, device)

In [ ]:
raw_test = json.load(open(p4["test"], encoding="utf-8"))
preds = predict_cases(student, raw_test, D, DOC_TAIL, device)          
write_cjpe_submission(preds, zip_name="submission_cjpe_kd.zip")